# Experiment 3: Zero-Cost Intra-Block Abort

This experiment implements an **Intra-Block Abort** mechanism that relies entirely on ReLU activation sparsity (Activity Level) to decide whether to skip the second convolution in a residual block.

Since this uses **zero extra parameters**, we can load the pre-trained `FP32` baseline directly and evaluate it zero-shot across different thresholds to see the latency/accuracy trade-off!

In [ ]:
!git clone https://github.com/MrRoboto11102003/Quantization_project.git 2>/dev/null; true
import sys
sys.path.append('Quantization_project')

import torch
import torchvision
import torchvision.transforms as transforms
import time, numpy as np, matplotlib.pyplot as plt

from experiment3 import activity_gated_resnet20

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

## Load Pre-trained Baseline

In [ ]:
model = activity_gated_resnet20().to(device)
BASELINE_PATH = 'Quantization_project/resnet20_base_best.pth'
model.load_state_dict(torch.load(BASELINE_PATH, map_location=device), strict=False)
model.eval()
print("Loaded Baseline FP32 successfully!")

## Zero-Shot Threshold Sweep
We sweep `tau` (the required fraction of non-zero ReLU activations to continue the block). 
If `tau = 0.0`, the model behaves exactly like the baseline (0% abort rate).
If `tau = 0.5`, the block aborts `Conv2` unless at least 50% of the `Conv1` outputs are non-zero.

In [ ]:
thresholds = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
results = []

for thresh in thresholds:
    correct, total = 0, 0
    total_flops_accum = 0
    
    # Measure Accuracy and FLOPs
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs, batch_flops = model(inputs, threshold=thresh)
            
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            total_flops_accum += batch_flops
            
    acc = 100. * correct / total
    flops_per_img = total_flops_accum / total
    
    # Measure Latency (30 runs)
    times = []
    with torch.no_grad():
        for _ in range(30):
            if torch.cuda.is_available(): torch.cuda.synchronize()
            start = time.time()
            for inputs, _ in testloader:
                inputs = inputs.to(device)
                _ = model(inputs, threshold=thresh)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            times.append(time.time() - start)
    lat = np.mean(times)
    
    results.append({
        'threshold': thresh,
        'acc': acc,
        'latency': lat,
        'flops': flops_per_img
    })
    
    flops_reduction = (1 - (flops_per_img / results[0]['flops'])) * 100
    print(f'Thresh: {thresh:4.2f} | Acc: {acc:5.2f}% | Latency: {lat:.4f}s | FLOPs: {flops_per_img/1e6:5.1f}M (Saved: {flops_reduction:4.1f}%)')

## Trade-off Visualizations

In [ ]:
accs = [r['acc'] for r in results]
lats = [r['latency'] for r in results]
flops = [r['flops']/1e6 for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(lats, accs, 'o-', color='#FF5722', linewidth=2, markersize=8)
axes[0].set_xlabel('Inference Latency (s)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy vs Latency Trade-off')
axes[0].grid(True, alpha=0.3)
for i, r in enumerate(results):
    axes[0].annotate(f"τ={r['threshold']}", (lats[i], accs[i]), textcoords="offset points", xytext=(0,5), ha='center')

axes[1].plot(flops, accs, 'o-', color='#4CAF50', linewidth=2, markersize=8)
axes[1].set_xlabel('Effective FLOPs (Millions)')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy vs Effective FLOPs')
axes[1].grid(True, alpha=0.3)
axes[1].invert_xaxis() # Lower FLOPs is better, so it's on the left

plt.tight_layout()
plt.savefig('activity_gating_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()